# Phase 2B: Extract MD&A from Local 10-K PDFs

Process real 10-K PDFs from `sec_filings_pdf/` folder and load MD&A chunks to PostgreSQL.

**Prerequisites:**
- Phase 2A completed (financial metrics loaded)
- 10-K PDFs in `sec_filings_pdf/` folder
- PostgreSQL running

In [3]:
import sys
sys.path.insert(0, '/Users/prajwalchambenandeeshappa/Github_Repos/Stocks_Earnings_Intelligence_Agent-Text2SQL/learning')

import os
import dlt
import logging
from pdf_mda_extractor import process_all_pdfs, prepare_chunks_for_dlt

# Set PostgreSQL credentials
os.environ['DESTINATION__POSTGRES__CREDENTIALS__HOST'] = 'localhost'
os.environ['DESTINATION__POSTGRES__CREDENTIALS__PORT'] = '5432'
os.environ['DESTINATION__POSTGRES__CREDENTIALS__DATABASE'] = 'financial_data'
os.environ['DESTINATION__POSTGRES__CREDENTIALS__USERNAME'] = 'postgres'
os.environ['DESTINATION__POSTGRES__CREDENTIALS__PASSWORD'] = 'postgres'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
print('✅ Imports successful')

✅ Imports successful


---
## Step 1: Extract MD&A from PDFs

In [4]:
# Process all 10-K PDFs and extract MD&A
filing_results = process_all_pdfs()

print(f"\n✅ Extraction complete: {len(filing_results)} PDFs processed")
print(f"   Total chunks to load: {sum(r['chunk_count'] for r in filing_results)}")

2026-07-12 13:22:23,381 - INFO - 
🚀 PHASE 2B: PDF MD&A Extraction
2026-07-12 13:22:23,382 - INFO - Found 47 PDF files

2026-07-12 13:22:23,382 - INFO - [1/47] amzn-20211231.pdf
2026-07-12 13:22:23,383 - INFO - 
2026-07-12 13:22:23,383 - INFO - 📄 Processing: AMZN 2021
2026-07-12 13:22:23,383 - INFO - ======================================================================
2026-07-12 13:22:23,384 - INFO -    Reading PDF...
2026-07-12 13:22:24,644 - INFO -    Total text: 264,049 characters
2026-07-12 13:22:24,644 - INFO -    Extracting MD&A...
2026-07-12 13:22:24,650 - INFO -    Extracted 162 characters of MD&A
2026-07-12 13:22:24,650 - INFO -    Chunking text...
2026-07-12 13:22:24,651 - INFO - ✅ Success! Extracted 1 chunks from 162 characters
2026-07-12 13:22:24,651 - INFO - [2/47] amzn-20221231.pdf
2026-07-12 13:22:24,651 - INFO - 
2026-07-12 13:22:24,651 - INFO - 📄 Processing: AMZN 2022
2026-07-12 13:22:24,651 - INFO - ====================================================================


✅ Extraction complete: 47 PDFs processed
   Total chunks to load: 47


---
## Step 2: Prepare Chunks for Loading

In [5]:
# Transform all chunks into dlt-ready format
all_chunks = []
for filing in filing_results:
    chunks = prepare_chunks_for_dlt(filing)
    all_chunks.extend(chunks)

print(f'📋 Prepared {len(all_chunks)} chunks for loading\n')

# Show sample
if all_chunks:
    print('Sample chunk:')
    sample = all_chunks[0]
    print(f"  Ticker: {sample['ticker']}")
    print(f"  Year: {sample['year']}")
    print(f"  Text length: {sample['text_length']} chars")
    print(f"  Preview: {sample['text'][:100]}...")

📋 Prepared 47 chunks for loading

Sample chunk:
  Ticker: AMZN
  Year: 2021
  Text length: 162 chars
  Preview: Item 7.Management’s Discussion and Analysis of Financial Condition and Results of Operations18Item 7...


---
## Step 3: Load to PostgreSQL

In [6]:
if all_chunks:
    # Create dlt pipeline
    pipeline = dlt.pipeline(
        pipeline_name='financial_data_pipeline',
        destination='postgres',
        dataset_name='sec_filings'
    )
    
    print('📝 Loading MD&A chunks to PostgreSQL...')
    pipeline.run(
        all_chunks,
        table_name='filing_text_chunks',
        write_disposition='replace'
    )
    print(f'✅ {len(all_chunks)} chunks loaded')
else:
    print('❌ No chunks to load')

2026-07-12 13:25:14,915|[INFO]|26550|8313057536|dlt|pipeline.py|_restore_state_from_destination:1724|The state was restored from the destination postgres (dlt.destinations.postgres):sec_filings


📝 Loading MD&A chunks to PostgreSQL...


2026-07-12 13:25:14,978|[INFO]|26550|8313057536|dlt|pool_runner.py|create_pool:203|Created none pool with 1 workers
2026-07-12 13:25:14,978|[INFO]|26550|8313057536|dlt|normalize.py|run:315|Running file normalizing
2026-07-12 13:25:14,978|[INFO]|26550|8313057536|dlt|normalize.py|run:318|Found 1 load packages
2026-07-12 13:25:14,981|[INFO]|26550|8313057536|dlt|normalize.py|run:341|Found 1 files in schema financial_data load_id 1783880714.9427981
2026-07-12 13:25:14,984|[INFO]|26550|8313057536|dlt|normalize.py|spool_schema_files:304|Created new load package 1783880714.9427981 on loading volume with 1 files
2026-07-12 13:25:14,987|[INFO]|26550|8313057536|dlt|worker.py|_get_items_normalizer:186|Created items normalizer JsonLItemsNormalizer with writer InsertValuesWriter for item format object and file format insert_values on table filing_text_chunks
2026-07-12 13:25:14,990|[INFO]|26550|8313057536|dlt|worker.py|w_normalize_files:284|Processed all items in 1 files
2026-07-12 13:25:14,990|[INF

✅ 47 chunks loaded


---
## Step 4: Verify All Tables

In [7]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='financial_data',
    user='postgres',
    password='postgres'
)
cursor = conn.cursor()

print('\n' + '='*70)
print('PHASE 2 COMPLETE - DATABASE SUMMARY')
print('='*70)

try:
    cursor.execute('SELECT COUNT(*) FROM sec_filings.financial_metrics')
    count = cursor.fetchone()[0]
    print(f'✅ financial_metrics: {count} records')
except:
    print('❌ financial_metrics not found')

try:
    cursor.execute('SELECT COUNT(*) FROM sec_filings.sec_filings_metadata')
    count = cursor.fetchone()[0]
    print(f'✅ sec_filings_metadata: {count} records')
except:
    print('❌ sec_filings_metadata not found')

try:
    cursor.execute('SELECT COUNT(*) FROM sec_filings.filing_text_chunks')
    count = cursor.fetchone()[0]
    print(f'✅ filing_text_chunks: {count} records')
except:
    print('❌ filing_text_chunks not found')

conn.close()

print('\n' + '='*70)
print('✅ PHASE 2 COMPLETE!')
print('Ready for Module 1: Agentic RAG')
print('='*70)


PHASE 2 COMPLETE - DATABASE SUMMARY
✅ financial_metrics: 180 records
✅ sec_filings_metadata: 15 records
✅ filing_text_chunks: 47 records

✅ PHASE 2 COMPLETE!
Ready for Module 1: Agentic RAG
